# M1: closing the two open gaps -- multiple comparisons, cross-session reliability

`01_explore.ipynb` closed with two concrete gaps before the beh-EEG
relationship could be called "safe and sound": (1) `correlation.py`'s
25-pair feature matrix wasn't corrected for multiple comparisons, and (2)
whether the relationship (spatial or individual-differences) is stable
across EEG sessions was untested. This notebook closes both -- honestly:
one gap closes cleanly, the other doesn't, and this notebook says so
rather than overstating what the data supports. See
`docs/ssvepbeh_reliability_gaps.md` for the full write-up and next steps.

In [1]:
import sys
sys.path.append('../../beh/scripts')
sys.path.append('../scripts')

import numpy as np
import pandas as pd

import loader as beh_loader
import overlap

sys.path.insert(0, '../scripts')
import plotting
import correlation
import session_reliability

analysis = overlap.analysis

beh_df = beh_loader.load_behavioral()
runmap_df = analysis.load_runmap()
baselines_df = analysis.load_baselines()
metadata_df = analysis.load_metadata()

categories = [
    ('HC', dict(group='CTR')),
    ('PD', dict(group='PD')),
    ('CVD', dict(group='CVD')),
    ('protan', dict(group='CVD', subgroup='protan')),
    ('deutan', dict(group='CVD', subgroup='deutan')),
]

## Gap 1: multiple-comparisons correction

`feature_correlations`' default feature sets give 5 x 5 = 25 pairwise
tests per group/pooled call. `correct_multiple_comparisons` applies Holm
(family-wise error rate, conservative) or Benjamini-Hochberg (FDR, more
power) via `statsmodels`.

In [2]:
feature_table = correlation.subject_features_table(beh_df, session=1)
pooled = correlation.feature_correlations(feature_table)
pooled_holm = correlation.correct_multiple_comparisons(pooled, method='holm')
pooled_fdr = correlation.correct_multiple_comparisons(pooled, method='fdr_bh')

pooled_fdr.sort_values('p_value')[['beh_feature', 'eeg_feature', 'r', 'p_value', 'p_corrected', 'significant']].head(6)

,beh_feature,eeg_feature,r,p_value,p_corrected,significant
9,beh_green,ramp_intercept,-0.375113,0.013192,0.106631,False
14,orientation_deg,ramp_intercept,-0.369224,0.014826,0.106631,False
12,orientation_deg,ramp_slope_red,0.353368,0.020101,0.106631,False
2,beh_red,ramp_slope_red,-0.350951,0.021029,0.106631,False
4,beh_red,ramp_intercept,0.347629,0.022363,0.106631,False
17,along_var,ramp_slope_red,0.340230,0.025591,0.106631,False


In [3]:
rows = []
for label, kw in [('pooled', {})] + categories:
    filt = {'group': kw.get('group'), 'subgroup': kw.get('subgroup')}
    r = correlation.feature_correlations(feature_table, **filt)
    c = correlation.correct_multiple_comparisons(r, method='fdr_bh')
    rows.append({'category': label, 'n_tests': len(c), 'min_p_value': c['p_value'].min(), 'min_p_corrected': c['p_corrected'].min(), 'any_significant': c['significant'].any()})
pd.DataFrame(rows)

,category,n_tests,min_p_value,min_p_corrected,any_significant
0,pooled,25,0.013192,0.106631,False
1,HC,25,0.059619,0.907857,False
2,PD,25,0.020599,0.257484,False
3,CVD,25,0.007011,0.175272,False
4,protan,25,0.114684,0.739397,False
5,deutan,25,0.003437,0.085916,False


**Nothing survives correction, pooled or in any group/subtype, under
either Holm or the more permissive FDR-BH.** The best pooled pair
(`orientation_deg` vs. `ramp_intercept`, uncorrected p=0.015) corrects to
p=0.11 under FDR -- not close. This gap does not close in the direction we
hoped: the individual-differences correlation looked promising in
`01_explore.ipynb`, but doesn't hold up once corrected for testing 25
hypotheses. **Read as a real negative result**, not a data-processing
artifact -- see the assessment at the end of this notebook.

## Gap 2: cross-session reliability

Is a finding stable when the EEG side is measured at a different session?
Restricted to subjects with EEG data at **both** sessions --
`session_reliability.paired_subjects` computes this directly from
`ssveps/files/subject_troughs.csv`.

In [4]:
paired_counts = pd.DataFrame([
    {'category': label, **{k: v for k, v in kw.items()}, 'n_paired': len(session_reliability.paired_subjects(**kw))}
    for label, kw in [('pooled', {})] + categories
])
paired_counts

,category,n_paired,group,subgroup
0,pooled,19,NaN,NaN
1,HC,13,CTR,NaN
2,PD,4,PD,NaN
3,CVD,2,CVD,NaN
4,protan,2,CVD,protan
5,deutan,0,CVD,deutan


**protan has only 2 subjects with EEG data at both sessions, CVD (combined)
also 2, and deutan has 0.** Per-subtype reliability -- the comparison that
actually matters most for the subtyping goal -- isn't assessable with
today's data at all. `session_reliability`'s functions raise a clear error
below `MIN_PAIRED_SUBJECTS=3` rather than silently reporting a number from
2 or fewer paired subjects.

In [5]:
for label, kw in [('pooled', {}), ('HC', dict(group='CTR')), ('PD', dict(group='PD'))]:
    try:
        session_reliability.session_overlap_comparison(beh_df, runmap_df, baselines_df, metadata_df, n_perm=200, seed=0, **kw)
    except ValueError as e:
        print(f'{label}: {e}')

for label, kw in [('CVD', dict(group='CVD')), ('protan', dict(group='CVD', subgroup='protan')), ('deutan', dict(group='CVD', subgroup='deutan'))]:
    try:
        session_reliability.session_overlap_comparison(beh_df, runmap_df, baselines_df, metadata_df, n_perm=200, seed=0, **kw)
        print(f'{label}: ran')
    except ValueError as e:
        print(f'{label}: {e}')

CVD: only 2 subjects have EEG data at both sessions for group='CVD' subgroup=None (need >= 3) -- not enough to assess reliability, see docs/ssvepbeh_reliability_gaps.md
protan: only 2 subjects have EEG data at both sessions for group='CVD' subgroup='protan' (need >= 3) -- not enough to assess reliability, see docs/ssvepbeh_reliability_gaps.md
deutan: only 0 subjects have EEG data at both sessions for group='CVD' subgroup='deutan' (need >= 3) -- not enough to assess reliability, see docs/ssvepbeh_reliability_gaps.md


### Spatial overlap: reliable where testable

In [6]:
for label, kw in [('pooled', {}), ('HC', dict(group='CTR')), ('PD', dict(group='PD'))]:
    r = session_reliability.session_overlap_comparison(beh_df, runmap_df, baselines_df, metadata_df, n_perm=5000, seed=0, **kw)
    r.insert(0, 'category', label)
    print(r.to_string(index=False))
    print()

category  session  n  weighted_overlap_obs  weighted_overlap_p  click_value_obs  click_value_p
  pooled        1 19              0.309978            0.009998         0.309978         0.0002
  pooled        2 19              0.306844            0.009998         0.306844         0.0002



category  session  n  weighted_overlap_obs  weighted_overlap_p  click_value_obs  click_value_p
      HC        1 13              0.306025            0.009998         0.306025         0.0002
      HC        2 13              0.305810            0.009998         0.305810         0.0002



category  session  n  weighted_overlap_obs  weighted_overlap_p  click_value_obs  click_value_p
      PD        1  4              0.238848            0.029794         0.238848         0.0002
      PD        2  4              0.206023            0.020996         0.206023         0.0002



**Highly stable.** For every category with enough paired subjects to test
(pooled n=19, HC n=13, PD n=4), `obs_stat`/`obs_mean` and p-values are
nearly identical between session 1 and session 2 on both tests. The spatial
overlap finding replicates cleanly -- this half of the EEG test's
justification is solid.

### Individual-differences correlation: not reliable where testable

In [7]:
for label, kw in [('pooled', {}), ('HC', dict(group='CTR')), ('PD', dict(group='PD'))]:
    r = session_reliability.session_correlation_comparison(beh_df, **kw)
    piv = r.pivot_table(index=['beh_feature', 'eeg_feature'], columns='session', values=['r', 'p_value'])
    piv.columns = [f'{a}_session{b}' for a, b in piv.columns]
    print(f'--- {label} ---')
    print(piv.sort_values('p_value_session1').head(4).to_string())
    print()

--- pooled ---
                              p_value_session1  p_value_session2  r_session1  r_session2
beh_feature eeg_feature                                                                 
beh_red     eeg_red                   0.020482          0.477722    0.526797    0.173411
perp_var    ramp_intercept            0.061469          0.059178   -0.436842   -0.440351
along_var   eeg_green                 0.226071          0.196927   -0.291424   -0.309711
beh_green   ramp_slope_green          0.266510          0.138663   -0.268421   -0.352632



--- HC ---
                            p_value_session1  p_value_session2  r_session1  r_session2
beh_feature eeg_feature                                                               
perp_var    ramp_intercept          0.034586          0.016537   -0.587912   -0.648352
            eeg_red                 0.057956          0.167672    0.537882    0.406861
along_var   eeg_green               0.086923          0.170386   -0.493011   -0.404504
beh_red     eeg_green               0.242912          0.843515   -0.348715   -0.060828



--- PD ---
                            p_value_session1  p_value_session2  r_session1  r_session2
beh_feature eeg_feature                                                               
along_var   eeg_green                    0.2          0.683772         0.8    0.316228
            eeg_red                      0.2          0.600000         0.8    0.400000
beh_red     ramp_intercept               0.2          0.200000         0.8    0.800000
            eeg_red                      0.2          0.400000         0.8    0.600000



/home/sebas/projects/DataAnalysis/.venv/lib/python3.13/site-packages/pingouin/power.py:862: UserWarning: Sample size is too small to estimate power (n <= 4). Returning NaN.
  warnings.warn("Sample size is too small to estimate power (n <= 4). Returning NaN.")
/home/sebas/projects/DataAnalysis/.venv/lib/python3.13/site-packages/pingouin/power.py:862: UserWarning: Sample size is too small to estimate power (n <= 4). Returning NaN.
  warnings.warn("Sample size is too small to estimate power (n <= 4). Returning NaN.")


**Not reliable.** Even in the categories with enough paired subjects to
compute at all, r-values shift substantially between sessions for the same
pair -- e.g. HC's `beh_red` vs. `eeg_green` drops from r=-0.35 (session 1)
to r=-0.06 (session 2); PD's `along_var` vs. `ramp_slope_red` drops from
r=0.6 to r=0.0. None of `01_explore.ipynb`'s headline pooled pairs
(`orientation_deg` vs. `ramp_slope_red`/`ramp_intercept`) even appear among
the top hits once restricted to this smaller paired subset (n=19 vs. the
main analysis's n=43) -- a second piece of evidence, independent of the
correction result above, that this finding isn't yet solid.

## Assessment: what's actually established, and what isn't

**Spatial overlap -- solid.** Significant in every group on two
independently-constructed null models (`01_explore.ipynb`), and stable
across EEG sessions everywhere it's testable. This part of the EEG test's
validity against behavior is well supported at this project's n.

**Individual-differences correlation -- not yet supported.** Looked
promising uncorrected, but (a) nothing survives multiple-comparisons
correction, pooled or per-group, under Holm or FDR, and (b) where
cross-session reliability is even computable, the correlations are
unstable session-to-session and don't reproduce the pooled analysis's
headline pairs. Two independent checks point the same direction: this
finding is not currently real evidence, whatever its uncorrected p-value
suggested.

**Why, per your own hypothesis going in:**

1. **Multidimensionality.** 25 univariate pairwise tests spend a lot of
   statistical budget without using the fact that the behavioral features
   (`beh_red`, `beh_green`, `orientation_deg`, `along_var`, `perp_var`) and
   EEG features (`eeg_red`, `eeg_green`, `ramp_slope_red`,
   `ramp_slope_green`, `ramp_intercept`) are each themselves correlated
   internally (e.g. `beh_red`/`beh_green` and `orientation_deg` are all
   derived from the same underlying click cloud). A joint/multivariate
   test that uses the covariance structure directly, rather than 25
   separate univariate ones, would have more power to detect a real but
   distributed relationship -- see next steps.
2. **Lack of points, especially per subtype.** protan/deutan/CVD-combined
   can't be assessed for reliability at all (2/0/2 paired subjects); even
   the categories that *can* be tested (pooled, HC, PD) sit at n=19/13/4
   once restricted to paired subjects -- thinner than the main analysis's
   n=43/21/6, and thin samples are exactly where a correlation's sign and
   magnitude are least stable, consistent with what session 1 vs. session
   2 actually showed above.

See `docs/ssvepbeh_reliability_gaps.md` for the concrete next steps this
points to (a multivariate approach, and how much more repeated-session
data each subtype would need).